# KIT KI-Toolbox: Test für den Kurs (Version 3)
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Die Zellen **von oben nach unten** ausführen. Jede Zelle meldet ✅ oder ❌ und sagt bei Problemen, was zu tun ist. Scheitert ein Schritt, nicht einfach weitermachen: Die folgenden Schritte überspringen sich dann selbst.

**Vorher einmalig**
1. Auf https://ki-toolbox.scc.kit.edu/ einen API-Schlüssel erstellen. Er beginnt mit `sk-`.
2. In Colab links auf das Schlüsselsymbol klicken, ein Secret mit dem Namen `KIT_TOOLBOX_KEY` anlegen, den Schlüssel als Wert eintragen und den **Notebook-Zugriff einschalten**.
3. Optional für Schritt 8: Secret `ANTHROPIC_API_KEY` für den Vergleich mit Claude.

Der Test schickt insgesamt etwa 40 kurze Anfragen.

## Schritt 1 · Installieren

In [ ]:
%pip install -q openai==3.24.0

## Schritt 2 · Schlüssel laden

In [ ]:
import os, json, time, asyncio, datetime
import pandas as pd
import openai
from openai import OpenAI, AsyncOpenAI
from pydantic import BaseModel, ValidationError
from typing import Literal

HINWEISE = {
    "SecretNotFoundError": "Es gibt kein Secret mit genau diesem Namen. Name prüfen: KIT_TOOLBOX_KEY",
    "NotebookAccessError": "Das Secret ist für dieses Notebook nicht freigegeben. Im Schlüssel-Menü den Notebook-Zugriff einschalten.",
}

def lies_secret(name, pflicht=True):
    try:
        from google.colab import userdata
    except ImportError:                                  # lokal: Umgebungsvariable
        wert = os.environ.get(name)
    else:
        try:
            wert = userdata.get(name)
        except Exception as e:
            if pflicht:
                print(f"❌ Secret {name}: {HINWEISE.get(type(e).__name__, type(e).__name__ + ': ' + str(e))}")
            return None
    return wert.strip() if wert else None

KEY = lies_secret("KIT_TOOLBOX_KEY")
if KEY and KEY.startswith("sk-"):
    print(f"✅ Schlüssel geladen: beginnt mit sk-, Länge {len(KEY)}")
elif KEY and KEY.startswith("eyJ"):
    print("❌ Das ist das Anmelde-Token (JWT), nicht der API-Schlüssel. Bitte einen API-Schlüssel erstellen.")
    KEY = None
elif KEY:
    print(f"⚠️ Schlüssel geladen, aber ungewöhnliches Format (Länge {len(KEY)}). Wir versuchen es trotzdem.")
else:
    print("❌ Kein Schlüssel geladen. Hinweis oben beachten, dann diese Zelle erneut ausführen.")

## Schritt 3 · Verbindung prüfen und Modelle auflisten

In [ ]:
BASE_URL = os.environ.get("KIT_TOOLBOX_BASE_URL", "https://ki-toolbox.scc.kit.edu/api/v1")
WUNSCH = ["kit.mistral-small-4-119b-a8b", "kit.deepseek-v4-flash", "kit.glm-5.3-flash"]
EMB_MODELL, RERANK_MODELL = "kit.qwen3-embedding-8b", "kit.qwen3-reranker-8b"
KEINE_CHATMODELLE = ["rerank", "embed", "flux", "whisper", "voxtral", "tts"]
ERGEBNISSE, MODELLE, IDS, BEREIT = [], [], [], False

def notiere(schritt, modell, ok, details="", sekunden=None):
    ERGEBNISSE.append({"schritt": schritt, "modell": modell, "ok": ok, "details": str(details)[:200],
                       "sekunden": None if sekunden is None else round(sekunden, 2)})
    print(f"{'✅' if ok else '❌'} {modell:24s} {'' if sekunden is None else f'{sekunden:5.1f} s'}  {str(details)[:110]}")

def fehlertext(e):
    return f"{type(e).__name__}: {getattr(e, 'message', None) or e}"[:200]

if not KEY:
    print("⏭ Erst Schritt 2 erfolgreich ausführen.")
else:
    client = OpenAI(api_key=KEY, base_url=BASE_URL, timeout=60, max_retries=1)
    t0 = time.time()
    try:
        ids = IDS = sorted(m.id for m in client.models.list())
        lokal = [i for i in ids if i.startswith("kit.")]
        chat = [i for i in lokal if not any(w in i for w in KEINE_CHATMODELLE)]
        MODELLE = [m for m in WUNSCH if m in ids] or chat[:3]
        BEREIT = bool(MODELLE)
        notiere("3", "Verbindung", True, f"{len(ids)} Modelle, davon {len(lokal)} am KIT gehostet", time.time() - t0)
        print("\nAm KIT gehostete Modelle:", ", ".join(lokal))
        print("Getestet werden:", ", ".join(MODELLE))
        fehlend = [m for m in WUNSCH if m not in ids]
        if fehlend:
            print("Nicht für deinen Schlüssel verfügbar:", ", ".join(fehlend))
    except openai.AuthenticationError as e:
        notiere("3", "Verbindung", False, "Server erreichbar, aber Schlüssel abgelehnt (401). Neuen API-Schlüssel erstellen.", time.time() - t0)
    except openai.APIConnectionError as e:
        notiere("3", "Verbindung", False, "Server nicht erreichbar. Lokal mit KIT-VPN wiederholen.", time.time() - t0)
    except Exception as e:
        notiere("3", "Verbindung", False, fehlertext(e), time.time() - t0)

def bereit():
    if not BEREIT:
        print("⏭ Übersprungen: erst Schritt 3 erfolgreich ausführen.")
    return BEREIT

## Schritt 4 · Einfacher Chat-Aufruf

In [ ]:
if bereit():
    for m in MODELLE:
        t0 = time.time()
        try:
            a = client.chat.completions.create(model=m, max_tokens=50,
                messages=[{"role": "system", "content": "Antworte so knapp wie möglich."},
                          {"role": "user", "content": "Antworte nur mit dem Wort: bereit"}])
            text = (a.choices[0].message.content or "").strip()
            tokens = f"{a.usage.prompt_tokens}+{a.usage.completion_tokens} Tokens" if a.usage else "keine Token-Angabe"
            notiere("4", m, "bereit" in text.lower(), f"{text[:40]!r}, {tokens}", time.time() - t0)
        except Exception as e:
            notiere("4", m, False, fehlertext(e), time.time() - t0)

## Schritt 5 · Tool Calling (wie in Woche 8)

Erfolg heißt: Das Modell ruft `get_machine_status` für M-03 auf **und** nutzt danach das Ergebnis in seiner Antwort.

In [ ]:
TOOLS = [{"type": "function", "function": {
    "name": "get_machine_status",
    "description": "Liefert den aktuellen Zustand einer Maschine der Nordwerk GmbH.",
    "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": "Maschinen-ID, z. B. M-03"}},
                   "required": ["machine_id"]}}}]
ERGEBNIS_M03 = {"id": "M-03", "name": "CNC-Drehmaschine", "status": "eingeschränkt", "hinweis": "Vibration erhöht"}

def tool_test(m):
    nachrichten = [{"role": "system", "content": "Du bist ein Instandhaltungs-Assistent. Nutze Tools, statt zu raten. Antworte auf Deutsch."},
                   {"role": "user", "content": "Wie ist der aktuelle Status der Maschine M-03?"}]
    a = client.chat.completions.create(model=m, max_tokens=300, messages=nachrichten, tools=TOOLS, tool_choice="auto")
    msg = a.choices[0].message
    if not msg.tool_calls:
        return False, f"kein Tool-Aufruf, Antwort: {(msg.content or '')[:70]!r}"
    call = msg.tool_calls[0]
    try:
        args = json.loads(call.function.arguments or "{}")
    except json.JSONDecodeError:
        return False, f"Argumente sind kein gültiges JSON: {call.function.arguments[:60]!r}"
    if call.function.name != "get_machine_status" or args.get("machine_id") != "M-03":
        return False, f"falscher Aufruf: {call.function.name}({args})"
    nachrichten.append({"role": "assistant", "content": msg.content or "",
                        "tool_calls": [{"id": call.id, "type": "function",
                                        "function": {"name": call.function.name, "arguments": call.function.arguments}}]})
    nachrichten.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(ERGEBNIS_M03, ensure_ascii=False)})
    b = client.chat.completions.create(model=m, max_tokens=300, messages=nachrichten, tools=TOOLS)
    antwort = b.choices[0].message.content or ""
    return "eingeschränkt" in antwort.lower(), f"Aufruf richtig, Antwort: {antwort[:70]!r}"

if bereit():
    for m in MODELLE:
        t0 = time.time()
        try:
            ok, info = tool_test(m)
            notiere("5", m, ok, info, time.time() - t0)
        except Exception as e:
            notiere("5", m, False, fehlertext(e), time.time() - t0)

## Schritt 6 · Strukturierte Ausgabe (wie in Woche 6)

Eine Störungsmeldung wird als JSON nach festem Schema klassifiziert. Zuerst streng per `json_schema`, falls abgelehnt per `json_object`.

In [ ]:
class Stoerung(BaseModel):
    maschine: str
    kategorie: Literal["mechanisch", "elektrisch", "hydraulisch", "sensorik", "sonstiges"]
    prioritaet: Literal["hoch", "mittel", "niedrig"]

SCHEMA = {"type": "object", "additionalProperties": False, "required": ["maschine", "kategorie", "prioritaet"],
          "properties": {"maschine": {"type": "string"},
                         "kategorie": {"type": "string", "enum": ["mechanisch", "elektrisch", "hydraulisch", "sensorik", "sonstiges"]},
                         "prioritaet": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]}}}
MELDUNG = "Spätschicht: An M-05 tritt Hydrauliköl am Zylinder aus, die Presse läuft noch, aber der Druck fällt langsam ab."

def struktur_test(m):
    nachrichten = [{"role": "system", "content": "Klassifiziere Störungsmeldungen. Antworte ausschließlich mit JSON mit den Feldern maschine, kategorie, prioritaet."},
                   {"role": "user", "content": MELDUNG}]
    try:
        a = client.chat.completions.create(model=m, max_tokens=200, messages=nachrichten,
            response_format={"type": "json_schema", "json_schema": {"name": "stoerung", "strict": True, "schema": SCHEMA}})
        modus = "json_schema"
    except openai.BadRequestError:
        a = client.chat.completions.create(model=m, max_tokens=200, messages=nachrichten, response_format={"type": "json_object"})
        modus = "json_object (Fallback)"
    text = (a.choices[0].message.content or "").strip().removeprefix("```json").removesuffix("```").strip()
    s = Stoerung.model_validate_json(text)
    richtig = s.maschine.upper().replace(" ", "") == "M-05" and s.kategorie == "hydraulisch"
    return richtig, f"{modus}: {s.model_dump()}"

if bereit():
    for m in MODELLE:
        t0 = time.time()
        try:
            ok, info = struktur_test(m)
            notiere("6", m, ok, info, time.time() - t0)
        except ValidationError as e:
            notiere("6", m, False, f"JSON passt nicht zum Schema: {str(e)[:100]}", time.time() - t0)
        except Exception as e:
            notiere("6", m, False, fehlertext(e), time.time() - t0)

## Schritt 7 · Zehn gleichzeitige Anfragen

Simuliert grob einen Moment im Kursabend, in dem viele gleichzeitig eine Zelle ausführen.

In [ ]:
async def eine_anfrage(ac, m, i):
    t0 = time.time()
    try:
        await ac.chat.completions.create(model=m, max_tokens=20, messages=[{"role": "user", "content": f"Nenne nur die Zahl {i}."}])
        return "ok", time.time() - t0
    except openai.RateLimitError:
        return "429", time.time() - t0
    except Exception as e:
        return type(e).__name__, time.time() - t0

if bereit():
    ac = AsyncOpenAI(api_key=KEY, base_url=BASE_URL, timeout=60, max_retries=0)
    t0 = time.time()
    erg = await asyncio.gather(*(eine_anfrage(ac, MODELLE[0], i) for i in range(10)))
    dauern = sorted(d for s, d in erg if s == "ok")
    zaehlung = pd.Series([s for s, _ in erg]).value_counts().to_dict()
    info = f"{zaehlung}" + (f", Median {dauern[len(dauern) // 2]:.1f} s, Maximum {dauern[-1]:.1f} s" if dauern else "")
    notiere("7", MODELLE[0], zaehlung.get("ok", 0) == 10, info, time.time() - t0)

## Schritt 8 · Embeddings und Reranker

8a prüft das Embedding-Modell: Zwei Sätze mit gleicher Bedeutung sollen ähnlicher sein als ein themenfremder Satz. 8b probiert, ob der Reranker über die API erreichbar ist. Das ist nur ein Versuch, ein ❌ ist hier kein Problem.

In [ ]:
import numpy as np, requests

if bereit():
    if EMB_MODELL in IDS:
        t0 = time.time()
        try:
            saetze = ["Die Spindel der Drehmaschine vibriert stark.",
                      "An der CNC-Drehmaschine treten heftige Schwingungen auf.",
                      "Die Mensa bietet heute Linsensuppe an."]
            e = client.embeddings.create(model=EMB_MODELL, input=saetze)
            v = [np.array(d.embedding) for d in e.data]
            cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
            aehnlich, fremd = cos(v[0], v[1]), cos(v[0], v[2])
            notiere("8a", EMB_MODELL, aehnlich > fremd, f"Dimension {len(v[0])}, ähnlich {aehnlich:.2f}, themenfremd {fremd:.2f}", time.time() - t0)
        except Exception as e:
            notiere("8a", EMB_MODELL, False, fehlertext(e), time.time() - t0)
    else:
        print("Embedding-Modell nicht in der Liste:", EMB_MODELL)

    if RERANK_MODELL in IDS:
        t0, gefunden = time.time(), None
        nutzlast = {"model": RERANK_MODELL, "query": "Warum vibriert die Drehmaschine?",
                    "documents": ["Unwucht im Spannfutter führt zu Vibration.", "Die Kantine schließt um 14 Uhr."]}
        for pfad in [BASE_URL + "/rerank", BASE_URL.rsplit("/v1", 1)[0] + "/rerank"]:
            try:
                r = requests.post(pfad, headers={"Authorization": f"Bearer {KEY}"}, json=nutzlast, timeout=30)
                print(f"{pfad} → HTTP {r.status_code}  {r.text[:100]}")
                if r.status_code == 200 and gefunden is None:
                    gefunden = pfad
            except Exception as e:
                print(f"{pfad} → {fehlertext(e)}")
        notiere("8b", RERANK_MODELL, gefunden is not None, f"Endpunkt: {gefunden or 'keiner gefunden'}", time.time() - t0)

## Schritt 9 · Optional: fünf Kontextfragen im Vergleich

Jede Frage bringt den nötigen Handbuch-Ausschnitt mit, wie bei RAG. Mit Secret `ANTHROPIC_API_KEY` läuft Claude Haiku 4.5 zum Vergleich mit (unter einem Cent).

In [ ]:
KONTEXT = """Wartungshandbuch Nordwerk GmbH (Auszug)
- M-03 CNC-Drehmaschine: Spindellager alle 500 Betriebsstunden nachschmieren. Bei Vibration über 4,5 mm/s Maschine stoppen.
- M-05 Hydraulikpresse: Hydrauliköl HLP 46, Ölwechsel alle 2.000 Betriebsstunden.
- M-07 Prüfstand: Schaltschranktemperatur maximal 45 °C. Elektrische Prüfung jährlich."""
FRAGEN = [("Wie oft wird das Spindellager von M-03 geschmiert?", ["500"]),
          ("Welches Öl braucht die Presse M-05?", ["HLP 46", "HLP46"]),
          ("Ab welcher Vibration muss M-03 gestoppt werden?", ["4,5", "4.5"]),
          ("Wie hoch darf die Schaltschranktemperatur am Prüfstand sein?", ["45"]),
          ("Wie oft wechselt man bei M-02 das Getriebeöl?", ["nicht", "keine"])]   # steht nicht im Kontext
SYSTEM_T8 = "Beantworte die Frage nur anhand des Kontexts. Steht die Antwort nicht darin, sage das."

def frage_kit(m, frage):
    a = client.chat.completions.create(model=m, max_tokens=150, messages=[
        {"role": "system", "content": SYSTEM_T8}, {"role": "user", "content": f"<kontext>{KONTEXT}</kontext>\n{frage}"}])
    return a.choices[0].message.content or ""

if bereit():
    kandidaten = {m: (lambda f, m=m: frage_kit(m, f)) for m in MODELLE}
    anth_key = lies_secret("ANTHROPIC_API_KEY", pflicht=False)
    if anth_key:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==1.11.0"], check=False)
        import anthropic
        anth = anthropic.Anthropic(api_key=anth_key)
        def frage_haiku(f):
            a = anth.messages.create(model="claude-haiku-4-5", max_tokens=150, system=SYSTEM_T8,
                                     messages=[{"role": "user", "content": f"<kontext>{KONTEXT}</kontext>\n{f}"}])
            return a.content[0].text
        kandidaten["claude-haiku-4-5"] = frage_haiku
    for name, fn in kandidaten.items():
        t0, treffer = time.time(), 0
        try:
            for frage, muss in FRAGEN:
                treffer += any(x.lower() in fn(frage).lower() for x in muss)
            notiere("9", name, treffer >= 4, f"{treffer} von {len(FRAGEN)} richtig", time.time() - t0)
        except Exception as e:
            notiere("9", name, False, fehlertext(e), time.time() - t0)

## Schritt 10 · Zusammenfassung

In [ ]:
if ERGEBNISSE:
    df = pd.DataFrame(ERGEBNISSE)
    namen = {"3": "3 Verbindung", "4": "4 Chat", "5": "5 Tool Calling", "6": "6 Strukturiert", "7": "7 Last", "8a": "8a Embeddings", "8b": "8b Reranker", "9": "9 Kontextfragen"}
    df["test"] = df["schritt"].map(namen)
    display(df.assign(ergebnis=df["ok"].map({True: "✅", False: "❌"}))
              .pivot_table(index="test", columns="modell", values="ergebnis", aggfunc="first").fillna(""))
    json.dump({"zeit": datetime.datetime.now().isoformat(timespec="seconds"), "endpunkt": BASE_URL, "ergebnisse": ERGEBNISSE},
              open("kit_toolbox_test.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    try:
        from google.colab import files
        files.download("kit_toolbox_test.json")
    except ImportError:
        print("gespeichert: kit_toolbox_test.json")

## Was die Ergebnisse bedeuten

| Ergebnis | Folgerung für den Kurs |
|---|---|
| Schritt 3 ✅ in Colab | Die Toolbox ist aus Colab nutzbar, kein VPN nötig. |
| Schritt 4 ✅ | Einfache Textaufgaben und Vergleichsübungen gehen über die Toolbox. |
| Schritt 5 ✅ | Tool Use und Agenten sind mit diesem Modell möglich, interessant für die Teamprojekte. |
| Schritt 5 ❌ | Wochen 8 und 10 bis 13 bleiben auf Claude. |
| Schritt 6 nur mit Fallback | Strukturierte Ausgaben gehen, aber ohne Garantie durch das Schema. Validierung bleibt Pflicht. |
| Schritt 7 mit 429 oder langen Wartezeiten | Vor dem Kurs mit dem SCC über Kontingente sprechen. |
| Schritt 8a ✅ | Embeddings können über die Toolbox laufen, das vereinfacht Woche 7. |
| Schritt 9 deutlich unter Claude | Die Toolbox eher als Spar- und Vergleichsoption nutzen, nicht als Hauptmodell. |